# Titanic Survival Prediction
Example of training a random forest classifier using Apache Spark's MLlib library.


In [0]:
import pandas as pd

from pyspark.sql.functions import col, median
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline

In [0]:
train_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

# 1. Pull data via pandas first, then convert it to a distributed Spark DataFrame
pdf = pd.read_csv(train_url)
df_spark = spark.createDataFrame(pdf)

# 2. View the distributed schema and display the first few rows
df_spark.printSchema()
display(df_spark.head(5))


In [0]:
# 1. Calculate the median age using Spark's approxQuantile tool
median_age = df_spark.approxQuantile("Age", [0.5], 0.001)[0]

# 2. Fill missing age and missing embarkation ports
df_cleaned = df_spark.na.fill({"Age": median_age, "Embarked": "S"})

# 3. Confirm missing values are resolved
display(df_cleaned.select("PassengerId", "Age", "Embarked").head(5))

In [0]:
# 1. Convert text strings ('Sex' and 'Embarked') into index numbers
sex_indexer = StringIndexer(inputCol="Sex", outputCol="SexIndex")
embarked_indexer = StringIndexer(inputCol="Embarked", outputCol="EmbarkedIndex")

# 2. Group all independent features into a single column named "features"
feature_cols = ["Pclass", "SexIndex", "Age", "SibSp", "Parch", "Fare", "EmbarkedIndex"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# 3. Define the Random Forest Model
rf = RandomForestClassifier(labelCol="Survived", featuresCol="features", numTrees=100, maxDepth=5)

# 4. Chain everything together into an end-to-end Pipeline
pipeline = Pipeline(stages=[sex_indexer, embarked_indexer, assembler, rf])

# 5. Split into 80% training and 20% testing data using Spark's randomSplit
train_data, test_data = df_cleaned.randomSplit([0.8, 0.2], seed=42)

# 6. Train the complete pipeline
model = pipeline.fit(train_data)

# 7. Make predictions on the testing slice
predictions = model.transform(test_data)
display(predictions.select("PassengerId", "Survived", "prediction", "probability").head(5))

In [0]:
evaluator = BinaryClassificationEvaluator(labelCol="Survived", rawPredictionCol="rawPrediction")
roc_auc = evaluator.evaluate(predictions)

print(f"SparkML Model ROC-AUC Accuracy: {roc_auc:.2%}")